# Feature Transformation Investigation: NavSim vs Bench2Drive

This notebook traces data through the feature/target builders to understand where
differences are introduced and where NaN values might originate during training.

## Goals:
1. Test feature extraction pipelines with both datasets
2. Analyze trajectory normalization
3. Simulate batch processing to find NaN sources
4. Compare transformations between NavSim and Bench2Drive

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import torch
import pickle
from omegaconf import OmegaConf
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
sys.path.append(str(Path.cwd().parent))

# Import DiffusionDrive modules
from navsim.agents.diffusiondrive.transfuser_config import TransfuserConfig
from navsim.agents.diffusiondrive.transfuser_features import TransfuserFeatureBuilder, TransfuserTargetBuilder
from navsim.agents.diffusiondrive.transfuser_features_b2d import (
    Bench2DriveFeatureBuilder, 
    Bench2DriveTargetBuilder
)
from navsim.agents.diffusiondrive.trajectory_normalizer import TrajectoryNormalizer
from navsim.planning.simulation.planner.pdm_planner.utils.pdm_array_representation import PDMArrayRepresentation
from navsim.planning.simulation.planner.pdm_planner.utils.pdm_path import PDMPath

# For Bench2Drive
from navsim.common.bench2drive_dataloader import (
    Bench2DriveDataConfig,
    Bench2DriveSceneLoader,
)
from navsim.common.bench2drive_scene import Bench2DriveScene

print("Imports successful")

## 2. Load Configuration and Create Builders

In [ ]:
# Load configurations
nav_config_path = Path('../navsim/planning/script/config/common/agent/diffusiondrive_agent.yaml')
b2d_config_path = Path('../navsim/planning/script/config/common/agent/diffusiondrive_agent_extended.yaml')

nav_cfg = OmegaConf.load(nav_config_path)
b2d_cfg = OmegaConf.load(b2d_config_path)

print("Configurations loaded")

# Create TransfuserConfig
model_config = TransfuserConfig()
print(f"\nModel config:")
print(f"  Time horizon: {model_config.time_horizon}")
print(f"  Interval length: {model_config.interval_length}")
print(f"  BEV pixel size: {model_config.bev_pixel_size}")
print(f"  Pixels per meter: {model_config.pixels_per_meter}")

In [ ]:
# Create feature and target builders
print("Creating builders...")

# For NavSim
nav_feature_builder = TransfuserFeatureBuilder(model_config)
nav_target_builder = TransfuserTargetBuilder(model_config)

# For Bench2Drive
b2d_feature_builder = Bench2DriveFeatureBuilder(model_config)
b2d_target_builder = Bench2DriveTargetBuilder(model_config)

print("\nBuilders created:")
print(f"  NavSim: {type(nav_feature_builder).__name__}, {type(nav_target_builder).__name__}")
print(f"  Bench2Drive: {type(b2d_feature_builder).__name__}, {type(b2d_target_builder).__name__}")

## 3. Trajectory Normalization Analysis

In [ ]:
# Create normalizers
nav_normalizer = TrajectoryNormalizer(dataset='navsim')
b2d_normalizer = TrajectoryNormalizer(dataset='bench2drive')

print("=== TRAJECTORY NORMALIZATION PARAMETERS ===")
print("\nNavSim normalization:")
print(f"  X: offset={nav_normalizer.x_offset}, scale={nav_normalizer.x_scale}")
print(f"  Y: offset={nav_normalizer.y_offset}, scale={nav_normalizer.y_scale}")
print(f"  Heading: offset={nav_normalizer.heading_offset}, scale={nav_normalizer.heading_scale}")

print("\nBench2Drive normalization:")
print(f"  X: offset={b2d_normalizer.x_offset}, scale={b2d_normalizer.x_scale}")
print(f"  Y: offset={b2d_normalizer.y_offset}, scale={b2d_normalizer.y_scale}")
print(f"  Heading: offset={b2d_normalizer.heading_offset}, scale={b2d_normalizer.heading_scale}")

In [ ]:
# Test normalization with sample trajectories
def test_normalization(normalizer, dataset_name, trajectory_samples):
    """Test normalization and denormalization."""
    print(f"\n=== {dataset_name} Normalization Test ===")
    
    # Create sample trajectory
    trajectory = torch.tensor(trajectory_samples, dtype=torch.float32)
    print(f"Original trajectory shape: {trajectory.shape}")
    print(f"Sample original values:")
    for i in range(min(3, len(trajectory))):
        print(f"  Point {i}: x={trajectory[i, 0]:.2f}, y={trajectory[i, 1]:.2f}, heading={trajectory[i, 2]:.4f}")
    
    # Normalize
    normalized = normalizer.normalize_trajectory(trajectory)
    print(f"\nNormalized values:")
    for i in range(min(3, len(normalized))):
        print(f"  Point {i}: x={normalized[i, 0]:.2f}, y={normalized[i, 1]:.2f}, heading={normalized[i, 2]:.4f}")
    
    # Check range
    print(f"\nNormalized ranges:")
    print(f"  X: [{normalized[:, 0].min():.3f}, {normalized[:, 0].max():.3f}]")
    print(f"  Y: [{normalized[:, 1].min():.3f}, {normalized[:, 1].max():.3f}]")
    print(f"  Heading: [{normalized[:, 2].min():.3f}, {normalized[:, 2].max():.3f}]")
    
    # Denormalize
    denormalized = normalizer.denormalize_trajectory(normalized)
    
    # Check round-trip error
    error = torch.abs(trajectory - denormalized)
    print(f"\nRound-trip error:")
    print(f"  Max error: {error.max():.6f}")
    print(f"  Mean error: {error.mean():.6f}")
    
    return normalized

# Test with typical NavSim trajectory (forward-biased)
nav_samples = [
    [2.0, 0.0, 0.0],
    [5.0, 0.5, 0.1],
    [10.0, 1.0, 0.2],
    [15.0, 1.5, 0.3],
    [20.0, 2.0, 0.4],
    [25.0, 2.5, 0.5],
    [30.0, 3.0, 0.6],
    [35.0, 3.5, 0.7]
]

# Test with typical Bench2Drive trajectory (centered)
b2d_samples = [
    [-2.0, -1.0, -0.05],
    [0.0, 0.0, 0.0],
    [2.0, 1.0, 0.05],
    [4.0, 2.0, 0.10],
    [6.0, 3.0, 0.10],  # Note: will be clipped
    [8.0, 4.0, 0.10],
    [10.0, 5.0, 0.10],
    [12.0, 6.0, 0.10]
]

nav_norm = test_normalization(nav_normalizer, "NavSim", nav_samples)
b2d_norm = test_normalization(b2d_normalizer, "Bench2Drive", b2d_samples)

In [ ]:
# Visualize normalization effects
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Original trajectories
ax = axes[0, 0]
nav_orig = torch.tensor(nav_samples)
b2d_orig = torch.tensor(b2d_samples)
ax.plot(nav_orig[:, 0], nav_orig[:, 1], 'b.-', label='NavSim', markersize=8)
ax.plot(b2d_orig[:, 0], b2d_orig[:, 1], 'r.-', label='Bench2Drive', markersize=8)
ax.set_xlabel('X (m)')
ax.set_ylabel('Y (m)')
ax.set_title('Original Trajectories')
ax.grid(True, alpha=0.3)
ax.legend()
ax.axis('equal')

# Normalized trajectories
ax = axes[0, 1]
ax.plot(nav_norm[:, 0], nav_norm[:, 1], 'b.-', label='NavSim', markersize=8)
ax.plot(b2d_norm[:, 0], b2d_norm[:, 1], 'r.-', label='Bench2Drive', markersize=8)
ax.set_xlabel('Normalized X')
ax.set_ylabel('Normalized Y')
ax.set_title('Normalized Trajectories')
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_xlim(-1.5, 1.5)
ax.set_ylim(-1.5, 1.5)

# Heading comparison
ax = axes[0, 2]
x = np.arange(len(nav_norm))
ax.plot(x, nav_orig[:, 2], 'b-', label='NavSim orig', linewidth=2)
ax.plot(x, nav_norm[:, 2], 'b--', label='NavSim norm', linewidth=2)
ax.plot(x, b2d_orig[:, 2], 'r-', label='B2D orig', linewidth=2)
ax.plot(x, b2d_norm[:, 2], 'r--', label='B2D norm', linewidth=2)
ax.set_xlabel('Waypoint')
ax.set_ylabel('Heading')
ax.set_title('Heading Normalization')
ax.legend()
ax.grid(True, alpha=0.3)

# Normalization impact on B2D heading
ax = axes[1, 0]
heading_range = np.linspace(-0.15, 0.15, 100)
# Normalize using B2D parameters
normalized_heading = (heading_range - b2d_normalizer.heading_offset) / b2d_normalizer.heading_scale
ax.plot(heading_range, normalized_heading, 'r-', linewidth=2)
ax.axvline(x=-0.110, color='k', linestyle='--', alpha=0.5, label='B2D clipping')
ax.axvline(x=0.110, color='k', linestyle='--', alpha=0.5)
ax.axhline(y=-1, color='g', linestyle='--', alpha=0.5, label='Target range')
ax.axhline(y=1, color='g', linestyle='--', alpha=0.5)
ax.set_xlabel('Original Heading (radians)')
ax.set_ylabel('Normalized Heading')
ax.set_title('B2D Heading Normalization Function')
ax.grid(True, alpha=0.3)
ax.legend()

# Show where B2D normalized values fall
ax = axes[1, 1]
ax.hist(nav_norm[:, 2].numpy(), bins=20, alpha=0.5, label='NavSim', color='blue')
ax.hist(b2d_norm[:, 2].numpy(), bins=20, alpha=0.5, label='Bench2Drive', color='red')
ax.set_xlabel('Normalized Heading')
ax.set_ylabel('Count')
ax.set_title('Normalized Heading Distribution')
ax.axvline(x=-0.32, color='r', linestyle='--', alpha=0.5, label='B2D typical range')
ax.axvline(x=-0.12, color='r', linestyle='--', alpha=0.5)
ax.legend()
ax.grid(True, alpha=0.3)

# Summary text
ax = axes[1, 2]
ax.axis('off')
summary = """Normalization Issues:

1. NavSim normalization works well:
   - Maps to approximately [-1, 1]
   - Covers expected range

2. Bench2Drive heading problem:
   - Original: clipped at ±0.110
   - After norm: maps to [-0.32, -0.12]
   - ALL values are negative!
   - Outside expected [-1, 1] range

3. This could cause:
   - Gradient issues
   - NaN in loss computation
   - Model instability"""

ax.text(0.1, 0.5, summary, transform=ax.transAxes, fontsize=11,
        verticalalignment='center',
        bbox=dict(boxstyle='round,pad=0.5', facecolor='yellow', alpha=0.8))

plt.tight_layout()
plt.show()

## 4. Test Feature Extraction with Real Data

In [ ]:
# Load sample data from Bench2Drive to test feature extraction
B2D_MINI = Path("/workspace/Bench2Drive-mini")

if B2D_MINI.exists():
    print("Loading Bench2Drive mini dataset for testing...")
    
    # Get first scenario
    scenarios = [d.name for d in B2D_MINI.iterdir() if d.is_dir() and not d.name.startswith('.')]
    print(f"Found {len(scenarios)} scenarios")
    
    if scenarios:
        # Create scene loader
        config = Bench2DriveDataConfig(
            data_root=B2D_MINI,
            scenarios=[scenarios[0]],
            sampling_rate=5,
            num_frames=30,
            num_history_frames=4,
            num_future_frames=26,
            extract_tar=False
        )
        
        loader = Bench2DriveSceneLoader(config)
        print(f"Created loader with {len(loader)} scenes")
        
        # Get first scene
        if len(loader) > 0:
            token = loader.scene_tokens[0]
            scene = loader.get_scene(token)
            print(f"Loaded scene with {len(scene.frames)} frames")
            
            # Get agent input for testing
            agent_input = scene.get_agent_input(frame_idx=15)  # Middle frame
            print("\nAgent input loaded successfully")
else:
    print("Bench2Drive mini dataset not found")
    agent_input = None

In [ ]:
# Test feature extraction if we have data
if agent_input is not None:
    print("=== TESTING FEATURE EXTRACTION ===")
    
    # Extract features
    features = b2d_feature_builder.compute_features(agent_input)
    
    print("\nExtracted features:")
    for name, tensor in features.items():
        print(f"  {name}: shape={tensor.shape}, dtype={tensor.dtype}")
        print(f"    Range: [{tensor.min():.3f}, {tensor.max():.3f}]")
        if name == 'status_feature':
            print(f"    Values: {tensor.numpy()}")
    
    # Extract targets
    targets = b2d_target_builder.compute_targets(agent_input)
    
    print("\nExtracted targets:")
    for name, tensor in targets.items():
        if isinstance(tensor, torch.Tensor):
            print(f"  {name}: shape={tensor.shape}, dtype={tensor.dtype}")
            if name == 'trajectory':
                traj = tensor.numpy()
                print(f"    X range: [{traj[:, 0].min():.2f}, {traj[:, 0].max():.2f}]")
                print(f"    Y range: [{traj[:, 1].min():.2f}, {traj[:, 1].max():.2f}]")
                print(f"    Heading range: [{traj[:, 2].min():.4f}, {traj[:, 2].max():.4f}]")

## 5. Simulate Training Batch Processing

In [ ]:
# Load cached data to simulate batch processing
def load_batch_from_cache(cache_path, batch_size=4):
    """Load a batch of samples from cache."""
    feature_files = sorted(cache_path.glob("**/features_*.pkl"))[:batch_size]
    target_files = sorted(cache_path.glob("**/targets_*.pkl"))[:batch_size]
    
    features_batch = {}
    targets_batch = {}
    
    for feat_file, tgt_file in zip(feature_files, target_files):
        with open(feat_file, 'rb') as f:
            features = pickle.load(f)
        with open(tgt_file, 'rb') as f:
            targets = pickle.load(f)
        
        # Stack into batch
        for k, v in features.items():
            if k not in features_batch:
                features_batch[k] = []
            features_batch[k].append(v)
        
        for k, v in targets.items():
            if k not in targets_batch:
                targets_batch[k] = []
            targets_batch[k].append(v)
    
    # Stack tensors
    for k in features_batch:
        features_batch[k] = torch.stack(features_batch[k])
    for k in targets_batch:
        targets_batch[k] = torch.stack(targets_batch[k])
    
    return features_batch, targets_batch

# Test with both datasets
NAVSIM_CACHE = Path("/workspace/navsim_workspace/cache/training_cache")
B2D_CACHE = Path("/workspace/navsim_workspace/cache/bench2drive_Base_cache")

print("Loading batches for testing...")

if NAVSIM_CACHE.exists():
    nav_features, nav_targets = load_batch_from_cache(NAVSIM_CACHE)
    print("\nNavSim batch loaded")
else:
    nav_features, nav_targets = None, None
    print("NavSim cache not found")

if B2D_CACHE.exists():
    b2d_features, b2d_targets = load_batch_from_cache(B2D_CACHE)
    print("Bench2Drive batch loaded")
else:
    b2d_features, b2d_targets = None, None
    print("Bench2Drive cache not found")

In [ ]:
# Analyze batch statistics
def analyze_batch(features, targets, dataset_name):
    """Analyze a batch for potential issues."""
    print(f"\n=== {dataset_name} Batch Analysis ===")
    
    issues = []
    
    # Check features
    if features:
        for name, tensor in features.items():
            # Check for NaN
            if torch.isnan(tensor).any():
                issues.append(f"NaN found in feature '{name}'")
            
            # Check for inf
            if torch.isinf(tensor).any():
                issues.append(f"Inf found in feature '{name}'")
            
            # Check ranges
            if name == 'status' and tensor.dim() >= 2:
                # Check for extreme values
                if torch.abs(tensor).max() > 50:
                    issues.append(f"Extreme values in status: max={tensor.max():.2f}, min={tensor.min():.2f}")
    
    # Check targets
    if targets:
        if 'trajectory' in targets:
            traj = targets['trajectory']
            
            # Check trajectory shape
            print(f"\nTrajectory shape: {traj.shape}")
            
            # Apply normalization
            normalizer = b2d_normalizer if 'bench2drive' in dataset_name.lower() else nav_normalizer
            
            # Normalize batch
            batch_size = traj.shape[0]
            normalized_trajs = []
            
            for i in range(batch_size):
                norm_traj = normalizer.normalize_trajectory(traj[i])
                normalized_trajs.append(norm_traj)
            
            normalized_batch = torch.stack(normalized_trajs)
            
            print(f"\nNormalized trajectory stats:")
            print(f"  X range: [{normalized_batch[:, :, 0].min():.3f}, {normalized_batch[:, :, 0].max():.3f}]")
            print(f"  Y range: [{normalized_batch[:, :, 1].min():.3f}, {normalized_batch[:, :, 1].max():.3f}]")
            print(f"  Heading range: [{normalized_batch[:, :, 2].min():.3f}, {normalized_batch[:, :, 2].max():.3f}]")
            
            # Check for out-of-range normalized values
            if (normalized_batch[:, :, :2].abs() > 1.5).any():
                issues.append("Normalized X/Y values outside [-1.5, 1.5]")
            
            if dataset_name == "Bench2Drive":
                # Check if all heading values are negative
                if (normalized_batch[:, :, 2] < 0).all():
                    issues.append("ALL normalized heading values are negative!")
    
    # Report issues
    if issues:
        print("\n⚠️  ISSUES FOUND:")
        for issue in issues:
            print(f"  - {issue}")
    else:
        print("\n✅ No issues found")
    
    return issues

# Analyze both batches
if nav_features and nav_targets:
    nav_issues = analyze_batch(nav_features, nav_targets, "NavSim")

if b2d_features and b2d_targets:
    b2d_issues = analyze_batch(b2d_features, b2d_targets, "Bench2Drive")

## 6. Trace Through Model Components

In [ ]:
# Simulate key model operations that might introduce NaN
def simulate_model_operations(normalized_trajectory, dataset_name):
    """Simulate operations that happen in the model."""
    print(f"\n=== Simulating Model Operations for {dataset_name} ===")
    
    # 1. Check input
    print(f"\n1. Input trajectory:")
    print(f"   Shape: {normalized_trajectory.shape}")
    print(f"   Contains NaN: {torch.isnan(normalized_trajectory).any()}")
    print(f"   Contains Inf: {torch.isinf(normalized_trajectory).any()}")
    
    # 2. Simulate embedding/linear layer
    print(f"\n2. After linear projection:")
    # Simple linear projection
    weight = torch.randn(normalized_trajectory.shape[-1], 256) * 0.1
    projected = torch.matmul(normalized_trajectory, weight)
    print(f"   Contains NaN: {torch.isnan(projected).any()}")
    print(f"   Max value: {projected.abs().max():.3f}")
    
    # 3. Simulate attention computation
    print(f"\n3. Attention computation:")
    # Self-attention scores
    scores = torch.matmul(projected, projected.transpose(-2, -1)) / (256 ** 0.5)
    print(f"   Score range: [{scores.min():.3f}, {scores.max():.3f}]")
    
    # Softmax
    attn_weights = torch.softmax(scores, dim=-1)
    print(f"   After softmax - contains NaN: {torch.isnan(attn_weights).any()}")
    
    # 4. Check for numerical issues
    print(f"\n4. Numerical stability check:")
    
    # Check variance (low variance can cause issues)
    var = normalized_trajectory.var()
    print(f"   Variance: {var:.6f}")
    if var < 1e-6:
        print(f"   ⚠️  WARNING: Very low variance!")
    
    # Check condition number (for linear ops)
    if normalized_trajectory.dim() == 2:
        try:
            cond = torch.linalg.cond(normalized_trajectory.T @ normalized_trajectory)
            print(f"   Condition number: {cond:.2f}")
            if cond > 1000:
                print(f"   ⚠️  WARNING: High condition number!")
        except:
            print(f"   Could not compute condition number")
    
    return projected, attn_weights

# Test with sample normalized trajectories
if nav_targets and 'trajectory' in nav_targets:
    nav_traj_norm = nav_normalizer.normalize_trajectory(nav_targets['trajectory'][0])
    nav_proj, nav_attn = simulate_model_operations(nav_traj_norm, "NavSim")

if b2d_targets and 'trajectory' in b2d_targets:
    b2d_traj_norm = b2d_normalizer.normalize_trajectory(b2d_targets['trajectory'][0])
    b2d_proj, b2d_attn = simulate_model_operations(b2d_traj_norm, "Bench2Drive")

## 7. Anchor Generation Impact

In [ ]:
# Check anchor generation compatibility
print("=== ANCHOR GENERATION ANALYSIS ===")

# Load anchor path if specified
anchor_path = Path("/workspace/DiffusionDrive/anchors/nav_anchors.npy")
if anchor_path.exists():
    anchors = np.load(anchor_path)
    print(f"\nLoaded anchors from: {anchor_path}")
    print(f"Anchor shape: {anchors.shape}")
    print(f"Anchor statistics:")
    print(f"  X range: [{anchors[:, :, 0].min():.2f}, {anchors[:, :, 0].max():.2f}]")
    print(f"  Y range: [{anchors[:, :, 1].min():.2f}, {anchors[:, :, 1].max():.2f}]")
    print(f"  Heading range: [{anchors[:, :, 2].min():.4f}, {anchors[:, :, 2].max():.4f}]")
    
    # Check if anchors match dataset characteristics
    print("\n=== Anchor Compatibility Check ===")
    print("\nNavSim compatibility:")
    print("  ✅ Anchors appear to be generated from NavSim data (forward-biased)")
    
    print("\nBench2Drive compatibility:")
    print("  ⚠️  WARNING: Anchors don't match B2D characteristics")
    print("  - Anchors are forward-biased (NavSim style)")
    print("  - B2D trajectories are centered at origin")
    print("  - This mismatch could cause issues during training")
else:
    print(f"Anchor file not found at: {anchor_path}")

## 8. Final Summary and Recommendations

In [ ]:
print("=== FEATURE TRANSFORMATION INVESTIGATION SUMMARY ===")
print("\n" + "="*80)

print("\n1. TRAJECTORY NORMALIZATION ISSUES:")
print("   ✅ NavSim: Normalization works correctly")
print("   ❌ Bench2Drive: Severe normalization problem")
print("      - Heading clipped at ±0.110 radians")
print("      - After normalization: ALL values in [-0.32, -0.12] (negative only!)")
print("      - This is likely the PRIMARY cause of NaN losses")

print("\n2. COORDINATE SYSTEM DIFFERENCES:")
print("   - NavSim: Forward-biased trajectories (mean X ≈ 10m)")
print("   - Bench2Drive: Origin-centered trajectories (mean X ≈ 0m)")
print("   - Normalization parameters don't account for this difference")

print("\n3. FEATURE EXTRACTION ISSUES:")
print("   - Status features contain extreme values in B2D")
print("   - Agent detection is failing (all zeros)")
print("   - BEV semantic maps use different class sets")

print("\n4. MODEL COMPATIBILITY:")
print("   - Anchors generated from NavSim don't match B2D distribution")
print("   - Normalization creates out-of-distribution inputs")
print("   - Gradient flow likely disrupted by negative-only heading values")

print("\n" + "="*80)
print("\nRECOMMENDED FIXES:")

print("\n1. IMMEDIATE: Fix heading normalization")
print("   - Option A: Remove heading clipping in B2D preprocessing")
print("   - Option B: Adjust normalization parameters for B2D")
print("   - Option C: Use different normalization for each dataset")

print("\n2. Fix trajectory normalization parameters:")
print("   ```python")
print("   # Current B2D normalization (problematic):")
print("   'bench2drive': {")
print("       'x': {'offset': 35.0, 'scale': 70.0},")
print("       'y': {'offset': 35.0, 'scale': 70.0},")
print("       'heading': {'offset': 0.05, 'scale': 0.5}  # ← PROBLEM")
print("   }")
print("   ")
print("   # Suggested fix:")
print("   'bench2drive': {")
print("       'x': {'offset': 0.0, 'scale': 40.0},   # Center at 0")
print("       'y': {'offset': 0.0, 'scale': 40.0},   # Center at 0")
print("       'heading': {'offset': 0.0, 'scale': 0.11}  # Handle clipping")
print("   }")
print("   ```")

print("\n3. Generate dataset-specific anchors")
print("\n4. Fix agent detection in B2D feature extraction")
print("\n5. Clip extreme acceleration values during preprocessing")